# Foreword

The Energy Transition Lectorate (ETL) asked us to create an energy classification label to help their clients make their homes greener. These labels are different from the government's labels because they are designed for people, not just houses. At first, Jesse and I tried to create these labels ourselves. After many attempts, we still weren't satisfied with the results. So, we asked Henri for help, and he gave us a formula to use.

In this notebook, I will gather all the necessary variables for the formula and create the labels based on these variables. For our project, everyone needed to make a predictive model. Jesse and I decided to create a decision tree for the energy classification label. We worked on the formula together, but we are doing the predictive models separately so we both get credit for it for making a predictive model.

To make this notebook easy to understand for as many people as possible, and because all other files are in English, this notebook will also be in English. This notebook is written by Mariëlle Groen, and this is my version of the decision tree model.

# Abstract
This notebook shows the development of an energy classification label for the Energy Transition Lectorate (ETL), aimed at making homes greener by considering both the house and its occupants. The project includes compiling necessary variables, applying a provided formula, creating labels out of these and making a decision tree model.

# Business Understanding

### Why?
The Energy Transition Lectorate (ETL) aims to create energy classification labels to help homeowners make their homes more sustainable. These labels will provide insights into energy consumption and potential cost savings from making their homes more energy-efficient. Unlike existing government labels, ETL's labels will be more accurate and consider both the physical characteristics of the house and the behavior of the occupants.

### Goals
- **Reduce Climate Change**: Contribute to the reduction of climate change by improving home energy efficiency, focusing on existing buildings which will make up the majority of houses by 2050.
- **Provide Actionable Advice**: Help homeowners understand their energy consumption, and how they can save money by making their homes more sustainable.
- **Improve Label Accuracy**: Develop more accurate energy classification labels that include a range of factors, such as the heat loss coefficient, to give better energy efficiency advice.

### Criteria for completion
- **Accuracy**: The labels accurately reflect the energy efficiency of the homes.
- **Clarity**: The labels are easy to understand
- **Client Feedback**: Positive feedback from ETL regarding the usefulness and accuracy of the labels.

# Data Understanding
I have done most data understanding in other files, which are really messy, but for this project, it is to late to do something about those files.

### Remeha and Need for heat
First we have the Remeha and Need For Heat (NFH) datasets. 
Both are set up like this: 
First: 'id', pseudo anonymized number to identify the client
Then: two rows I don't understand and we dont need to understand
After that: 'timestamp' date and time of the measurement
Second to last and lastly: 'property' and 'value', property tells you what kind of value it is, and value is said value belonging to the property.

From remeha we use the inside temperature ('temp_in__degC'), because the temperatures from NFH weren't accurate. The temperature sensor measured the temperatures always a bit too low, and it had gaps in the data for some clients.
The variable used from NFH is cummulative gas usage ('g_use_cum__m3'). Here we use NFH because this data was the overal gas usage, and not for specific parts.

### Weather
The database "weather" is a csv file made from the HistoricDutchWeather/KNMI database. Eric made for the time period needed a new database because use of HistoricDutchWeather used up a lot of time and it was quite difficould to use. It has all the things that the HistoricDutchWeather had too.

### H__W_K_1
This is the csv-file Eric made from the H__W_K_1 he calculated per house. It has the id's, H__W_K_1-values and the R^2 of the calculated H__W_K_1.

# Data preparation

In [ ]:
# Imports
import pandas as pd 
from datetime import datetime
import numpy as np
from ipywidgets import interact, Dropdown
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier 
from sklearn import tree
import matplotlib.pyplot as plt


needforheat_file_path='needforheat_export_student.parquet'
remeha_file_path='remeha_export_student.parquet'

In [ ]:
# Fixed variables
label_GHI__J_m_2_a_1 = 3588 * 10^6
h_d_1 = 24
W_kW_1 = 1000
s_h_1 = 3600
label_DHW__kWh_a_1_p_1 = 856
A_sol__m2 = 16.8

In [ ]:
%%time
# Attempt to read the Parquet file remeha
try:
    df_remeha = pd.read_parquet(
        remeha_file_path,
        engine='pyarrow',
        use_nullable_dtypes=True  #Outcommented because of futureWarning
        #dtype_backend='numpy_nullable'
    )
    print("File was successfully read without specifying compression codec.")
except Exception as e:
    print(f"Error reading file: {e}")
    
# For me (dtype_backend='numpy_nullable') didn't work and (use_nullable_dtypes=True) did, for others it was the oppesite.
# If the import doesn't work for you, switch the hashtag to the other mentioned line.

In [ ]:
%%time
# Attempt to read the Parquet file needforheat
try:
    df_nfh= pd.read_parquet(
        needforheat_file_path,
        engine='pyarrow',
        use_nullable_dtypes=True #Outcommented because of futureWarning
        #dtype_backend='numpy_nullable'
    )
    print("File was successfully read without specifying compression codec.")
except Exception as e:
    print(f"Error reading file: {e}")
    
# For me (dtype_backend='numpy_nullable') didn't work and (use_nullable_dtypes=True) did, for others it was the oppesite.
# If the import doesn't work for you, switch the hashtag to the other mentioned line.

In [ ]:
# Making copies
df_nfh = df_nfh.copy()
df_remeha = df_remeha.copy()

In [ ]:
# Reset index
df_nfh.reset_index(inplace=True)
df_remeha.reset_index(inplace=True)

In [ ]:
# Extract only the id values which are in both dataframes
common_ids = set(df_nfh['id']).intersection(set(df_remeha['id']))

# Filter dataframes to include only common ids
df_nfh_filtered = df_nfh[df_nfh['id'].isin(common_ids)]
df_remeha_filtered = df_remeha[df_remeha['id'].isin(common_ids)]

# Extract necessary columns from df_nfh
df_nfh_filtered = df_nfh_filtered[df_nfh_filtered['property'] == 'g_use_cum__m3'][['timestamp', 'id', 'value']]

# Extract necessary columns from df_remeha
df_remeha_filtered = df_remeha_filtered[df_remeha_filtered['property'] == 'temp_in__degC'][['timestamp', 'id', 'value']]

# Confert datetime to date
df_nfh_filtered['timestamp'] = pd.to_datetime(df_nfh_filtered['timestamp'], utc=True, errors='coerce')
df_nfh_filtered['timestamp'] = df_nfh_filtered['timestamp'].dt.tz_convert('Europe/Amsterdam')
df_nfh_filtered['date'] = df_nfh_filtered['timestamp'].dt.date
df_remeha_filtered['timestamp'] = pd.to_datetime(df_remeha_filtered['timestamp'], utc=True, errors='coerce')
df_remeha_filtered['timestamp'] = df_remeha_filtered['timestamp'].dt.tz_convert('Europe/Amsterdam')
df_remeha_filtered['date'] = df_remeha_filtered['timestamp'].dt.date

# Group by 'date' and calculate the mean of 'value' for df_remeha
df_remeha_daily = df_remeha_filtered.groupby(['id', 'date'])['value'].mean().reset_index()
df_remeha_daily = df_remeha_daily.rename(columns={"value": "temp_in_degC"})

# Group by 'date' and min the 'value' for df_nfh
df_nfh_daily_min = df_nfh_filtered.groupby(['id', 'date'])['value'].min().reset_index()

# Group by 'date' and calculate the difference between consecutive days for g_use_cum__m3
df_nfh_daily_min['value_diff'] = df_nfh_daily_min.groupby('id')['value'].diff().fillna(0)
df_nfh_daily_min = df_nfh_daily_min.rename(columns={"value_diff": "g_use__m3_daily"})

# Merge on id and date
df_combi = pd.merge(df_remeha_daily, df_nfh_daily_min, how="inner", on=["id", "date"])
df_combi.drop(['value'], axis=1, inplace = True)

# Display the resulting DataFrame
print(df_combi.info()) 
print(df_combi)

In [ ]:
# Read the CSV file, make it read the unnamed column as a datetime value, rename the unnamed column
weather = pd.read_csv('weather.csv', parse_dates=['Unnamed: 0'])
weather.rename(columns={'Unnamed: 0': 'date'}, inplace=True)

# Convert the 'date' column to datetime in UTC
weather['date'] = pd.to_datetime(weather['date'], utc=True, errors='coerce')

# Convert the datetime to the 'Europe/Amsterdam' timezone
weather['date'] = weather['date'].dt.tz_convert('Europe/Amsterdam')

# Extract the date part (discarding the time)
weather['date'] = weather['date'].dt.date

# Group by the 'date' column and calculate the mean for each day
daily_mean_weather = weather.groupby('date').mean().reset_index()

In [ ]:
#read Heatlosscoeficient
H__W_K_1 = pd.read_csv('df_H__W_K_1.csv')

In [ ]:
# make df floor area m2
floor_area__m2 = {
    'id': [573579, 538728, 590793, 522131, 528230, 545968, 557614, 504509, 526833, 558926,
           579485, 562321, 586195, 581277, 542663, 565038, 528213, 538163, 579082, 565396,
           591006, 559286, 504607],
    'floor_area__m2': [186, 165, 123, 105, 129, 167, 171, 103, 135, 131, 156, 155, 82, 188, 81, 173, 144, 174,
                       170, 140, 213, 112, 110]
}
floor_area__m2 = pd.DataFrame(floor_area__m2)
floor_area__m2.head()

In [ ]:
# Merge DataFrames on 'id'
merged_df = pd.merge(H__W_K_1, floor_area__m2, on='id', how='inner')

# Select the required columns
data = merged_df[['id', 'H__W_K_1', 'floor_area__m2']]
data.head()

In [ ]:
# Group by 'id' and calculate the mean of 'temp_in_degC'
mean_temp_in_degC = df_combi.groupby('id')['temp_in_degC'].mean()

# Transform the mean values
label_temp_demand_K_d = (mean_temp_in_degC - 10.6716667) * 365

# Convert the result to a DataFrame
label_temp_demand_K_d = label_temp_demand_K_d.reset_index()
label_temp_demand_K_d.columns = ['id', 'label_temp_demand_K_d']

# Merge the result back into the original data DataFrame
data = data.merge(label_temp_demand_K_d, on='id', how='left')
data.head()

In [ ]:
# Calculate and include label_CH__kWh_a_1
data["label_CH__kWh_a_1"] = (data["H__W_K_1"] * data["label_temp_demand_K_d"] / h_d_1 - 
                         A_sol__m2 * label_GHI__J_m_2_a_1 / s_h_1) / W_kW_1
data.head()

In [ ]:
# Calculate and include label_CH__kWh_a_1_m_2
data["label_CH__kWh_a_1_m_2"] = data["label_CH__kWh_a_1"]  / data["floor_area__m2"]
data.head()

In [ ]:
# create a list of our conditions
conditions = [
    (data["floor_area__m2"] <= 30),
    (data["floor_area__m2"] > 30) & (data["floor_area__m2"] <= 100),
    (data["floor_area__m2"] > 100)
    ]

# create a list of the values we want to assign for each condition
values = [1, (2.28 - 1.18/70 * (100 - data["floor_area__m2"])), (1.28 + 0.01 * data["floor_area__m2"])]

# create a new column and use np.select to assign values to it using our lists as arguments
data["label_occupancy__p"] = np.select(conditions, values)

data.head(10)

In [ ]:
data["label_DHW__kWh_a_1_m_2"] = ((label_DHW__kWh_a_1_p_1 * data["label_occupancy__p"]) / data["floor_area__m2"])
data.head(10)

In [ ]:
data["label__kWh_a_1_m_2"] = data["label_CH__kWh_a_1_m_2"] + data["label_DHW__kWh_a_1_m_2"]
data.head(18)

In [ ]:
# Calculate what would be a wonderfull situation and what would be the worst. The static numbers used aren't perfect, 
# they are just to get some kind of range of how high or low the label__kWh_a_1_m_2 can get, to be able to rate the labels accordingly
# As for variable names, I used the names used before, just with g_ before the name for the good values and b_ for the bad ones.
# For the H__W_K_1 I found this, these wonderfull H__W_K_1-values are made possible by ChatGPT:
    #Heel slecht:
    #180 W/m²K: Dit is erg slecht en betekent dat de woning veel warmte verliest. Woningen met deze U-waarde zijn waarschijnlijk niet goed geïsoleerd en hebben hoge stookkosten.
    #Voldoende:
    #120 - 180 W/m²K: Dit is redelijk, maar er is nog ruimte voor verbetering. Woningen met deze U-waarde kunnen baat hebben van extra isolatiemaatregelen.
    #Goed:
    #80 - 120 W/m²K: Dit is een goede score voor een woning die is gebouwd volgens de huidige normen. Woningen met deze U-waarde verliezen relatief weinig warmte en hebben lagere stookkosten.
    #Zeer goed:
    #< 80 W/m²K: Dit is een zeer goede score voor een woning die is gebouwd met extra isolatiemaatregelen. Woningen met deze U-waarde verliezen zeer weinig warmte en hebben zeer lage stookkosten.
    #Voorbeelden:
    #Een slecht geïsoleerde vrijstaande woning uit de jaren 70: 200 W/m²K of meer
    #Een goed geïsoleerde rijwoningen uit de jaren 90: 100 W/m²K
    #Een nieuwbouwwoning met moderne isolatie: 60 W/m²K of lager

#good:
g_m2 = 500 #The internet said 500 m2 would be seen as a huge house.
g_label_temp_demand_K_d = (16 - 10.6716667) * 365 #According to the internet, lower than 16*C inside could cause health isues.
g_label_occupancy__p = 1.28 + 0.01 * g_m2
g_H__W_K_1 = 60

#Calculations for the good label
g_label_CH__kWh_a_1 = (g_H__W_K_1 * g_label_temp_demand_K_d / h_d_1 - A_sol__m2 * label_GHI__J_m_2_a_1 / s_h_1) / W_kW_1
g_label_CH__kWh_a_1_m_2 = g_label_CH__kWh_a_1  / g_m2
g_label_DHW__kWh_a_1 = label_DHW__kWh_a_1_p_1 * g_label_occupancy__p
g_label_DHW__kWh_a_1_m_2 = g_label_DHW__kWh_a_1  / g_m2
g_label__kWh_a_1_m_2 = g_label_CH__kWh_a_1_m_2 + g_label_DHW__kWh_a_1_m_2
print(g_label__kWh_a_1_m_2)

#bad
b_m2 = 30 #In excel file "REDUCEDHEATCARB-field-data-collection" page "NL_home_size_overview" the biggest size catogory in used for single famaly homes was 15 to 50 m2, I assumed somewere between those two numbers would be small.
b_label_temp_demand_K_d = (25 - 10.6716667) * 365 #I just used 25, you can go higher, but where do you stop going higher. In my opinion 25*C is a hot temperature to have inside a house. 
b_label_occupancy__p = 1.28 + 0.01 * b_m2
b_H__W_K_1 = 200

#Calculations for the bad value.
b_label_CH__kWh_a_1 = (b_H__W_K_1 * b_label_temp_demand_K_d / h_d_1 - A_sol__m2 * label_GHI__J_m_2_a_1 / s_h_1) / W_kW_1
b_label_CH__kWh_a_1_m_2 = b_label_CH__kWh_a_1  / b_m2
b_label_DHW__kWh_a_1 = label_DHW__kWh_a_1_p_1 * b_label_occupancy__p
b_label_DHW__kWh_a_1_m_2 = b_label_DHW__kWh_a_1  / b_m2
b_label__kWh_a_1_m_2 = b_label_CH__kWh_a_1_m_2 + b_label_DHW__kWh_a_1_m_2
print(b_label__kWh_a_1_m_2)

In [ ]:
# Calculate spreiding
# I used a higher number to devide them by, because it would make the range between the labels smaller and you would get a bad label faster. Before you would only get a bad label if you lived in a carton box.
spreiding = (b_label__kWh_a_1_m_2 - g_label__kWh_a_1_m_2) / 25

# Convert the DataFrame column to a numpy array for conditions
label_kWh_array = data["label__kWh_a_1_m_2"].to_numpy()

# Define conditions
conditions = [
    (label_kWh_array <= g_label__kWh_a_1_m_2),
    (label_kWh_array > g_label__kWh_a_1_m_2) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 2),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 2) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 3),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 3) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 4),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 4) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 5),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 5) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 6),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 6) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 7),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 7) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 8),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 8) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 9),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 9) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 10),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 10) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 11),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 11) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 12),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 12) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 13),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 13) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 14),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 14) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 15),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 15) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 16),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 16) & (label_kWh_array <= g_label__kWh_a_1_m_2 + spreiding * 17),
    (label_kWh_array > g_label__kWh_a_1_m_2 + spreiding * 17)
]

#Create a list of the values we want to assign for each label. 
#I made a lot of labels, so you would see more difference between the houses. 
#Otherwise they would all be devided between A and B, and you wouldn't really see the difference between the houses.
values = ['S', 'S-', 'A+', 'A',  'A-', 'B+', 'B',  'B-', 'C+', 'C', 'C-', 'D+', 'D',  'D-', 'E+', 'E', 'E-', 'F+', 'F']

# Create a new column and use np.select to assign values to it using our lists as arguments
data["label"] = np.select(conditions, values)

# Display the first few rows of the dataframe
print(data.head(18))
print(data.info())

# Modelling

In [ ]:
# Features and target
X = data.drop(columns=['id', 'label', 'label__kWh_a_1_m_2', 'label_CH__kWh_a_1', 'label_CH__kWh_a_1_m_2', 'label_DHW__kWh_a_1_m_2'])
y = data['label']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create a decision tree classifier
clf = DecisionTreeClassifier(random_state=42)

# Train the classifier
clf.fit(X_train, y_train)

# Plot the decision tree
plt.figure(figsize=(25, 10))
tree.plot_tree(clf, feature_names=X.columns, class_names=clf.classes_, filled=True)
plt.show()

# Evaluation
### Labels

Why does ID 542663 have such a low rating, as it is the only one with a C+? The house has an H__W_K_1 of 127.959528 and an area of 81 m², while for example, 559286 has an H__W_K_1 of 123.294527 and an area of 112 m², and 559286 has a rating of B-. 542663 loses about the same amount of heat while their house is much smaller.

If you look at 581277, which is probably a unique case in and of itself, it has an H__W_K_1 of 336.630668, the highest in our dataset, and an area of 188 m². The only reason why 581277 has a rating of A- is because they have a large area. Since they have an enormous heat loss, I would think they deserve a lower rating and label. Personally, I think the area should be removed from the formula because currently, a large part of the rating is based on the area, which is not fair.

### Decision tree biggest impact

label_occupancy__p is seen as the biggest impact on the decision tree, which is for a big part based of the floor area. And floor area is seen as the second biggest devider. Based of this I can say that the floor area could almost be seen as a sole contributer to the label given to the house.

### Criteria for completion met?
- **Accuracy:** Criteria accuracy is not met, the labels do not reflect energie efficiency at all.
- **Clarity:** Criteria clarity is met, the labels are easy to understand. Maybe the letter S is not that universally used outside of games, but it stands for superior. When told S would be the best label, it should be clear.
- **Client Feedback:** Criteria client feedback is not met. ETL agreed with using letters like the currently existing labels, but I am not sure about the way I used them. We also have not had a meeting about the accuracy of the currant labels. But they have mentioned not liking the use of floor area being promenent, so I asume they agree in my conclution of not finding the labels usefull.

# Conclution
The labeling system for energy efficiency ratings we've made is flawed, particularly in how it uses floor area. We should rewrite the formula used to create the energy efficiency ratings and make sure floor area is not the main contributor to the outcoming label. The current clarity of the labels is good, make sure this stays clear when revisiting evaluation. The criteria for completion should be able to stay the same. The energy efficiency ratings are complete when it meets this set criteria.